# SmolVLM-500M-Instruct — DIMER image captioning and VQA tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/smolvlm-vision-language-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/smolvlm-vision-language-pipeline/blob/main/tutorials/smolvlm_vision_language_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-HuggingFaceTB%2FSmolVLM--500M--Instruct-ffcc4d?style=flat)](https://huggingface.co/HuggingFaceTB/SmolVLM-500M-Instruct)
[![Upstream](https://img.shields.io/badge/Upstream-huggingface%2Fsmollm-181717?style=flat&logo=github&logoColor=white)](https://github.com/huggingface/smollm)
[![arXiv](https://img.shields.io/badge/arXiv-2504.05299-b31b1b.svg)](https://arxiv.org/abs/2504.05299)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** image + text to text generation (captioning and visual question answering) using the pinned SmolVLM-500M-Instruct weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`SmolVLMPipeline`) rather than reimplementing model inference. At inference one image and one user prompt are wrapped in the model's chat template, the processor resizes the image to a longest edge of 2048 px and splits it into 512 px tiles (64 visual tokens each), and the Idefics3-architecture model — a SigLIP-derived vision encoder feeding the SmolLM2-360M-Instruct text decoder, which is itself already a DIMER language-model profile — generates the assistant turn. Decoding is **greedy by default** (`do_sample=False`, `DECODING = "greedy"`), so a rerun on the same device, dtype and library versions reproduces the same text; `do_sample=True` is exposed for callers who want varied wording and gives up that determinism. The output is **free text with no score, no probability and no correctness signal**: the model writes fluent prose whether or not it is right — the model card's smoke run correctly named a red square but also claimed its corners touched the image edges, which they did not — so every answer must be read as a hypothesis. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What upstream supplies is the model, processor and chat template; what this repository adds is manifest verification, input validation and ceilings, prompt assembly, and a fixed output contract with `new_tokens`/`truncated` run facts.

**Learning objectives:** bootstrap the repository in a fresh runtime, generate a synthetic image (or upload your own), surface the pipeline's ceilings and the decoding contract, stage and digest-verify the immutable upstream snapshot, run a captioning prompt and a question prompt through the public API with explicit generation settings, read the output contract correctly (including the truncation flag), understand why no metric is reported and what labelled data a real evaluation needs, and export the answers plus provenance.

**This notebook does not demonstrate:** multi-image or video input (`MAX_IMAGES` = 1), object detection or grounding with coordinates, OCR with layout, text-only chat, batched inference, fine-tuning, or any accuracy claim. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (bfloat16 there, the dtype the checkpoint ships in); on the model card's CPU smoke the snapshot loaded in 5.7 s and 128 new tokens took 17.4 s, so the two-prompt default takes about a minute on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 1.0 GB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python and PIL image handling; what greedy decoding is and why free text has no intrinsic accuracy.
- **Data:** the default sample is a synthetic image generated in code; BYOD is one image file, gated off by default. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded images remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Git clone, the pinned wheel installs, and the fetch of the missing snapshot file from the Hugging Face Hub at the immutable revision. No credentials are needed.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `huggingface-hub`, `safetensors`, `numpy`, `pillow`) are directly pinned by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on CPU and bfloat16 on CUDA (the checkpoint's shipped dtype), so generated text can differ between the two device paths; no compilation or quantization is applied.

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/smolvlm-vision-language-pipeline.git'
REPO_NAME = 'smolvlm-vision-language-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, PIL, numpy, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pillow': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a 384 x 384 white canvas drawn in this cell with a filled red square in the upper left and a filled blue circle in the lower right — so it needs no download, contains no personal data, and is reproducible from code (no randomness, no seed; its pixel SHA-256 is printed and exported). Two prompts are asked about it: a captioning prompt and a counting question. The drawing has **no reference answers** the notebook asserts, so every answer it produces is smoke/sanity evidence that the code path works — you can judge the answers by eye, but that is a reading, not a measurement, and the model card's smoke run shows how a fluent answer can be wrong in detail.

BYOD is optional and disabled by default. Expected BYOD input: exactly one image file that Pillow can open (PNG, JPEG, WebP, ...), any mode (converted to RGB), with both sides between 1 and `MAX_IMAGE_SIDE` = 4096 px; edit the `PROMPTS` form field (one prompt per `|`, each at most `MAX_TEXT_CHARS` characters) to ask your own questions. The upload stays inside this runtime.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
PROMPTS = 'Describe this image in one sentence. | How many shapes are in the image, and what colour is each one?'  # @param {type:"string"}
prompts = [prompt.strip() for prompt in PROMPTS.split('|') if prompt.strip()]
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f'upload exactly one image, got {len(uploaded)}')
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    sample_kind = 'BYOD upload'
else:
    # Deterministic drawing: a red square and a blue circle on white.
    image = Image.new('RGB', (384, 384), (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.rectangle((48, 48, 176, 176), fill=(220, 30, 30))
    draw.ellipse((208, 208, 336, 336), fill=(30, 60, 220))
    image_name = 'synthetic_square_circle_384'
    sample_kind = 'synthetic (drawn in this cell)'
sample_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample': image_name, 'sample_kind': sample_kind, 'mode': image.mode, 'size': image.size, 'prompts': prompts, 'pixel_sha256': sample_sha256})

## 3. Validate the input against the pipeline ceilings and state the decoding contract

The pipeline enforces its ceilings through constants imported here from the package so the values shown are the ones in force: `MAX_IMAGES` (exactly one image per call), `MAX_IMAGE_SIDE` (either side, px), `MAX_TEXT_CHARS` (prompt length), `MAX_NEW_TOKENS` (hard ceiling on `max_new_tokens`; `DEFAULT_MAX_NEW_TOKENS` is the default) and `DECODING` (the default decoding mode). This cell surfaces them and checks the image and prompts before any model work, naming the failing condition and the corrective action; `generate()` re-applies the same rules authoritatively and raises `TypeError`/`ValueError` on its own. **What the pipeline changes about your image:** the processor resizes it so the longest edge is 2048 px (aspect ratio preserved) and splits it into 512 px tiles, each becoming 64 visual tokens; nothing is cropped away. A `max_new_tokens` budget of 96 is used per prompt below; an answer that uses the whole budget is reported as `truncated` — a cut-off answer, not a complete one.

In [ ]:
from smolvlm_vision_language_pipeline import DECODING, DEFAULT_MAX_NEW_TOKENS, MAX_IMAGE_SIDE, MAX_IMAGES, MAX_NEW_TOKENS, MAX_TEXT_CHARS

ceilings = {'MAX_IMAGES': MAX_IMAGES, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING}
MAX_NEW_TOKENS_PER_PROMPT = 96
print(ceilings)
problems = []
width, height = image.size
if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
    problems.append(f'image side {image.size} outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: resize the image and rerun Section 2')
if not prompts:
    problems.append('PROMPTS is empty: enter at least one prompt')
for index, prompt in enumerate(prompts):
    if len(prompt) > MAX_TEXT_CHARS:
        problems.append(f'prompt {index} has {len(prompt)} chars > MAX_TEXT_CHARS={MAX_TEXT_CHARS}: shorten it')
if not 1 <= MAX_NEW_TOKENS_PER_PROMPT <= MAX_NEW_TOKENS:
    problems.append(f'MAX_NEW_TOKENS_PER_PROMPT must be within 1..MAX_NEW_TOKENS={MAX_NEW_TOKENS}')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
print({'width': width, 'height': height, 'images_per_call': 1, 'model_input': 'longest edge 2048 (aspect preserved), 512-px tiles', 'decoding': f'{DECODING} (do_sample=False), max_new_tokens={MAX_NEW_TOKENS_PER_PROMPT}', 'within_ceilings': True})

## 4. Stage, verify, and resolve the pinned model

The public API pins the exact upstream model repository and immutable 40-hex revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here) and loads with `trust_remote_code=False` through the native `transformers` Idefics3 classes. The repository commits the DIMER snapshot manifest (`weights/smolvlm-500m-instruct/dimer-base-manifest.json`: byte sizes and SHA-256 digests of every snapshot file) and the small config/processor/tokenizer files, but git-ignores the 1.0 GB `model.safetensors`, so a fresh clone must stage that file first. The package's `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches only the manifest-listed files that are absent, from the Hub at the pinned revision, into the repository's weights directory, and returns the list it fetched (`['model.safetensors']` on a fresh clone, `[]` when everything is already staged); it refuses to stage if the committed manifest disagrees with the package's pinned identity. `verify_snapshot()` then re-hashes every listed file and raises on the first size or digest mismatch; only afterwards does `from_pretrained` build processor and model from that verified directory with `local_files_only=True` (`source: local-snapshot`). The effective model identity, the selected device and the effective dtype are printed before inference.

In [ ]:
from smolvlm_vision_language_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, SmolVLMPipeline, stage_missing_files, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'fetched': fetched, 'from': MODEL_ID, 'revision': MODEL_REVISION})
snapshot_info = verify_snapshot(WEIGHTS_DIR)
print({'snapshot_path': snapshot_info['path'], 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')})
pipe = SmolVLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'dtype': pipe.dtype, 'source': pipe.source})

## 5. Generate answers and interpret them

Each call to `generate(images, prompt, *, max_new_tokens=..., do_sample=False)` returns `text` (the decoded assistant turn, special tokens stripped), the `prompt`, `image_size`, `new_tokens`, `truncated` (true when the answer used the whole `max_new_tokens` budget and was cut off), the `generation` settings actually used (`max_new_tokens`, `do_sample`, `decoding`), device, dtype, source and model identity. **Output semantics:** `text` is free-form generated language with **no score and no correctness signal**; a fluent, specific answer is not evidence that it is right. With greedy decoding the same inputs reproduce the same text on a fixed device, dtype and library version — that is a reproducibility property, not a quality one.

**Evaluation:** the repository ships **no metric helper and reports no performance measure**, because open-ended generation has no intrinsic correctness signal. A real evaluation needs labelled data matched to the use: question-answer pairs with reference answers for VQA accuracy, reference captions for a caption metric such as CIDEr, or document pages with gold answers for document-QA exact match — and the caller's own scoring code over enough items to state a dispersion. The synthetic drawing has none, so **no metric is reported**; you can compare the answers with what you see (a red square and a blue circle), but that reading is a sanity check on one drawing, not a measurement, and the model card's smoke run is the reminder that plausible detail can be invented. The sanity checks below are plumbing checks (text returned, settings as requested); the truncation flag tells you whether an answer was cut. The runtime figures are measured on the runtime identified in Section 1 for this image and include the first-call warm-up.

In [ ]:
import time

answers = []
for prompt in prompts:
    started = time.perf_counter()
    result = pipe.generate(image, prompt, max_new_tokens=MAX_NEW_TOKENS_PER_PROMPT, do_sample=False)
    answers.append({'prompt': prompt, 'text': result['text'], 'new_tokens': result['new_tokens'], 'truncated': result['truncated'], 'generation': result['generation'], 'seconds': round(time.perf_counter() - started, 3)})
    print({'prompt': prompt, 'seconds': answers[-1]['seconds'], 'new_tokens': result['new_tokens'], 'truncated': result['truncated'], 'generation': result['generation']})
    print('answer:', result['text'])
checks = {
    'one_answer_per_prompt': len(answers) == len(prompts),
    'answers_are_text': all(isinstance(a['text'], str) and a['text'].strip() for a in answers),
    'greedy_as_requested': all(a['generation']['do_sample'] is False and a['generation']['decoding'] == DECODING for a in answers),
    'budget_as_requested': all(a['generation']['max_new_tokens'] == MAX_NEW_TOKENS_PER_PROMPT for a in answers),
}
if not all(checks.values()):
    raise RuntimeError(f'generate output failed a sanity check: {checks}')
print({'checks': checks, 'any_truncated': any(a['truncated'] for a in answers)})
metrics = {}
print('no metric is reported: free-text answers have no intrinsic correctness signal and the repository ships no metric helper; evaluate on labelled question-answer pairs or reference captions')

## 6. Export answers and provenance

One JSON record is written under `outputs/`: an `answers` list with, per prompt, the prompt, the generated text, `new_tokens`, `truncated`, the generation settings and seconds (so every answer maps back to its prompt and the image), the sanity checks, the ceilings in force, the empty metric block, the sample identity (name, kind, size, pixel digest), the repository revision, the model identifier and immutable revision, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, Pillow, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
payload = {
    'answers': [{'index': index, **answer} for index, answer in enumerate(answers)],
    'sanity_checks': checks,
    'ceilings': ceilings,
    'metrics': metrics,
    'sample': {'name': image_name, 'kind': sample_kind, 'width': image.width, 'height': image.height, 'pixel_sha256': sample_sha256},
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'snapshot': {'path': snapshot_info['path'], 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'pillow': PIL.__version__,
        'device': pipe.device,
        'dtype': pipe.dtype,
    },
}
with open('outputs/smolvlm_vision_language_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))
print('outputs/smolvlm_vision_language_result.json')

## Interpretation and limits

The answers are generated language, not measurements: they carry no score, no probability and no signal of correctness, and the model can state details that are not in the image — the model card's smoke run named a red square correctly and then invented that its corners touched the edges. On the synthetic drawing the answers are plumbing evidence only; no metric is reported because none exists without labelled question-answer pairs or reference captions, and a real evaluation needs such a set in your domain plus your own scoring code. The pipeline accepts one image and one prompt per call, resizes the longest edge to 2048 px and tiles it, caps answers at `MAX_NEW_TOKENS` (a `truncated` answer is cut, not complete), and exposes no grounding coordinates, no OCR layout, no text-only chat and no batching. Greedy decoding is deterministic on a fixed device and dtype (CPU float32 and CUDA bfloat16 can produce different text); `do_sample=True` trades that determinism for varied wording. The text decoder, SmolLM2-360M-Instruct, is the same model behind the DIMER language-model profile of that name, so its language-side limits (small model, English-centric instruction tuning) apply here too.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot, validate the demonstrated input against the enforced ceilings, execute the public pipeline path with explicit greedy generation settings, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, captioning or VQA accuracy on any domain, freedom from hallucination, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/smolvlm-500m-instruct/` and rerun Section 4. A `ValueError` naming `MAX_IMAGE_SIDE`, `MAX_TEXT_CHARS` or the image count in Section 2 or 3: fix the BYOD input or the `PROMPTS` field and rerun from Section 2. `truncated: True` on an answer: raise `MAX_NEW_TOKENS_PER_PROMPT` in Section 3 (up to `MAX_NEW_TOKENS`). Slow generation on a CPU runtime is expected (about 0.14 s per token on the card's machine).

**Next experiments.** Upload a photograph and ask a question whose answer you know, then ask a question whose answer is not in the image and watch whether the model declines or invents one; rerun the default prompts with `do_sample=True` to see wording vary while the greedy run stays fixed; run the same prompts on a CUDA runtime and diff the bfloat16 answers against the CPU float32 ones. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/HuggingFaceTB/SmolVLM-500M-Instruct
- Upstream code: https://github.com/huggingface/smollm
- SmolVLM paper: https://arxiv.org/abs/2504.05299
- Text decoder (DIMER language-model profile): https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct